In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from scipy.special import expit
import time

# ---------------------------------------------------------------------------
# Step 1: Setup Environment
# ---------------------------------------------------------------------------
warnings.filterwarnings("ignore")
random_state = 42

# ---------------------------------------------------------------------------
# Step 2: Define Constants
# ---------------------------------------------------------------------------
survival_methylation_list = [
    'cg27423208', 'cg14928902', 'cg01783151', 'cg00624834', 'cg02135682',
    'cg18946117', 'cg03803002', 'cg07909527', 'cg24636541', 'cg15421962',
    'cg25800170', 'cg08521684', 'cg18276638', 'cg19546356', 'cg03330642',
    'cg05128038', 'cg19058453', 'cg00917251', 'cg20460234', 'cg21586412',
    'cg11876574', 'cg09208611', 'cg19952015', 'cg12360736', 'cg07251193',
    'cg17625506', 'cg16477886', 'cg16337339', 'cg01513611', 'cg25281562',
    'cg25457956', 'cg26885220', 'cg23508333', 'cg20961943', 'cg08349335',
    'cg13996963', 'cg07770866', 'cg10547005', 'cg02295509', 'cg20498859',
    'cg05765605', 'cg17904988', 'cg18803303', 'cg14189750', 'cg24173629',
    'cg14584031', 'cg26391832', 'cg04939408', 'cg15283904', 'cg11481582',
    'cg15262984', 'cg06914505', 'cg12078728', 'cg02093006', 'cg13252540',
    'cg25281042', 'cg05638299', 'cg01890712', 'cg13206302', 'cg21001641',
    'cg19472079', 'cg04285130', 'cg08477744', 'cg12416080', 'cg05252940',
    'cg02162897', 'cg00395063', 'cg08015253', 'cg08110500', 'cg15965055',
    'cg23916044', 'cg09624244', 'cg11278602', 'cg25820223', 'cg07991229',
    'cg03206537', 'cg00031303', 'cg26344024', 'cg21629653', 'cg14445366',
    'cg08599300', 'cg05585947', 'cg24425972', 'cg22376081', 'cg10760320',
    'cg11017746', 'cg09640070', 'cg20773956', 'cg20222376', 'cg22853371',
    'cg09200308', 'cg02369195', 'cg11146034', 'cg12250896', 'cg27165687',
    'cg01663232', 'cg26874282', 'cg07302069', 'cg08187687', 'cg20334627',
    'cg21952115', 'cg03020916', 'cg16185831', 'cg25591868', 'cg20158867',
    'cg25673357', 'cg06821992', 'cg15460348', 'cg05791136', 'cg00063174',
    'cg01765545', 'cg11075509', 'cg06170754', 'cg02411088', 'cg02519415',
    'cg02878891', 'cg27660920', 'cg10246976', 'cg05943293', 'cg24012925',
    'cg00460795', 'cg15174393', 'cg10639981', 'cg24603152', 'cg00219585',
    'cg09320113', 'cg18170141', 'cg23509344', 'cg23118464', 'cg22412747',
    'cg17536595', 'cg15481370', 'cg03964958', 'cg19132462', 'cg01203443',
    'cg06281425', 'cg21045770', 'cg21624854', 'cg11930554', 'cg03214765',
    'cg03821987', 'cg09944012', 'cg25644556', 'cg05695927', 'cg00394793',
    'cg10599438', 'cg17023971', 'cg22865720', 'cg19952015', 'cg00792723',
    'cg07239592', 'cg26184856', 'cg02833116', 'cg25100604', 'cg07517487',
    'cg27353899', 'cg02994066', 'cg26958558', 'cg26417874', 'cg22163199',
    'cg10947764', 'cg04348250', 'cg04807594', 'cg19247841', 'cg10275917',
    'cg11530693', 'cg05387167', 'cg18014547', 'cg20984502', 'cg05682719',
    'cg23429678', 'cg04257969', 'cg13088374', 'cg11868356', 'cg04187088',
    'cg23933345', 'cg22365350', 'cg02470039', 'cg22273168'
]

MGMT_cols = ['cg12434587', 'cg12981137']

# ---------------------------------------------------------------------------
# Step 3: Data Loading Function
# ---------------------------------------------------------------------------
def load_data(base_path):
    """
    Loads clinical, label, GBM, and LGG data files.
    Checks that each file exists before attempting to load.
    """
    file_paths = {
        "clinical_data": os.path.join(base_path, "2019_TCGA-CDR-SupplementalTableS1.xlsx"),
        "new_labels": os.path.join(base_path, "Matrix_WHO2021.csv"),
        "gbm_data": os.path.join(base_path, "GBM_450K_Filtered_X-Y,SNPs,Non-overlap850K.csv"),
        "lgg_data": os.path.join(base_path, "LGG-450K_Filtered_X-Y,SNPs,Non-overlap850K.csv"),
    }
    
    for key, path in file_paths.items():
        if not os.path.exists(path):
            print(f"❌ Error: Missing {key} file: {path}")
            return None
    print("📂 Loading data...")
    
    clinical_df = pd.read_excel(file_paths["clinical_data"])
    new_labels = pd.read_csv(file_paths["new_labels"])
    gbm_df = pd.read_csv(file_paths["gbm_data"])
    lgg_df = pd.read_csv(file_paths["lgg_data"])
    
    print("✅ Data loaded successfully!")
    return clinical_df, new_labels, gbm_df, lgg_df

# ---------------------------------------------------------------------------
# Step 4: Preprocess Methylation Data
# ---------------------------------------------------------------------------
def preprocess_methylation_data(df):
    """
    Transposes the methylation DataFrame so that the first row becomes the header.
    """
    df_trns = df.T
    df_trns.columns = df_trns.iloc[0]
    return df_trns[1:]

# ---------------------------------------------------------------------------
# Step 5: Merge Clinical and Methylation Data
# ---------------------------------------------------------------------------
def merge_data(clinical_df, new_labels, gbm_df, lgg_df):
    """
    Merges clinical and methylation data:
    - Renames the clinical barcode column.
    - Filters for GBM and LGG.
    - Merges clinical data with new labels.
    - Preprocesses and concatenates GBM and LGG methylation data.
    - Merges methylation and clinical data.
    """
    
    clinical_df.rename(columns={'bcr_patient_barcode': 'Patient_ID'}, inplace=True)
    filtered_clinical = clinical_df[clinical_df['type'].isin(['GBM', 'LGG'])]
    merged_clinical = pd.merge(filtered_clinical, new_labels, on='Patient_ID', how='inner')
    gbm_processed = preprocess_methylation_data(gbm_df)
    lgg_processed = preprocess_methylation_data(lgg_df)
    
    merged_methylation = pd.concat([gbm_processed, lgg_processed])
    merged_methylation.rename(columns={'Index': 'Patient_ID'}, inplace=True)
    
    merged_data = pd.merge(merged_methylation, merged_clinical, left_index=True, right_on='Patient_ID', how='inner')
    print(f"✅ Merged Data Shape: {merged_data.shape}")
    return merged_data

# ---------------------------------------------------------------------------
# Step 6: Calculate MGMT Probability
# ---------------------------------------------------------------------------
def calculate_mgmt(df):
    """
    Converts the two MGMT probe columns to numeric, calculates M-values,
    computes a linear predictor, and applies the inverse logit function.
    Removes the original probe columns and intermediate calculations.
    """
    for col in MGMT_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    if df[MGMT_cols].isnull().any().any():
        print("Warning: Some beta values could not be converted to numeric and are set to NaN.")
    
    df['M1'] = np.log2(df[MGMT_cols[0]] / (1 - df[MGMT_cols[0]]))
    df['M2'] = np.log2(df[MGMT_cols[1]] / (1 - df[MGMT_cols[1]]))
    df['y'] = 4.3215 + 0.5271 * df['M1'] + 0.9265 * df['M2']
    df['MGMT'] = expit(df['y'])
    
    df.drop(columns=MGMT_cols + ['M1', 'M2', 'y'], inplace=True)
    print("✅ MGMT calculation complete.")
    return df

# ---------------------------------------------------------------------------
# Step 7: Filter Out Unclassified Cancers
# ---------------------------------------------------------------------------
def filter_unclassified(df):
    """
    Removes rows where the classification label is 'unclassified'.
    """
    if 'classification.2021_simplified.labels' in df.columns:
        df = df[df['classification.2021_simplified.labels'].str.lower() != 'unclassified']
    print("✅ Unclassified cancers removed.")
    return df

# ---------------------------------------------------------------------------
# Step 8: Subset Final Data
# ---------------------------------------------------------------------------
def subset_data(df):
    """
    Selects only the metadata, survival methylation probes, and MGMT columns.
    """
    metadata_cols = ['Patient_ID', 'OS.time', 'OS', 'classification.2021_simplified.labels']
    methylation_cols = [col for col in survival_methylation_list if col in df.columns]
    final_cols = metadata_cols + methylation_cols + ['MGMT']
    subset_df = df[final_cols]
    print("✅ Data subset complete.")
    return subset_df

# ---------------------------------------------------------------------------
# Step 9: Main Workflow with Progress Prints
# ---------------------------------------------------------------------------
def main():
    start_time = time.time()
    base_path = "/Users/nijat/Downloads/OneDrive_1_2024-09-21/Data"
    
    # Load data
    data = load_data(base_path)
    if data is None:
        return
    print("Step 1 complete: Data loaded.")
    print("Time elapsed: {:.2f} seconds".format(time.time() - start_time))
    
    clinical_df, new_labels, gbm_df, lgg_df = data
    merged_df = merge_data(clinical_df, new_labels, gbm_df, lgg_df)
    print("Step 2 complete: Data merged.")
    print("Time elapsed: {:.2f} seconds".format(time.time() - start_time))
    
    merged_df = calculate_mgmt(merged_df)
    print("Step 3 complete: MGMT calculated.")
    print("Time elapsed: {:.2f} seconds".format(time.time() - start_time))
    
    merged_df = filter_unclassified(merged_df)
    print("Step 4 complete: Unclassified cancers filtered out.")
    print("Time elapsed: {:.2f} seconds".format(time.time() - start_time))
    
    final_df = subset_data(merged_df)
    print("Step 5 complete: Data subsetted.")
    print("Time elapsed: {:.2f} seconds".format(time.time() - start_time))
    
    print(f"Final DataFrame Shape: {final_df.shape}")
    print(final_df.head())
    
    output_path = "/Users/nijat/Desktop/Final/subset_features_data.csv"
    final_df.to_csv(output_path, index=False)
    print(f"Step 6 complete: Final data saved to '{output_path}'.")
    print("Total time elapsed: {:.2f} seconds".format(time.time() - start_time))
    
if __name__ == "__main__":
    main()


📂 Loading data...
✅ Data loaded successfully!
Step 1 complete: Data loaded.
Time elapsed: 13.93 seconds
✅ Merged Data Shape: (653, 403991)
Step 2 complete: Data merged.
Time elapsed: 29.09 seconds
✅ MGMT calculation complete.
Step 3 complete: MGMT calculated.
Time elapsed: 31.10 seconds
✅ Unclassified cancers removed.
Step 4 complete: Unclassified cancers filtered out.
Time elapsed: 32.87 seconds
✅ Data subset complete.
Step 5 complete: Data subsetted.
Time elapsed: 32.87 seconds
Final DataFrame Shape: (619, 184)
       Patient_ID  OS.time   OS classification.2021_simplified.labels  \
368  TCGA-14-1043     24.0  1.0                          glioblastoma   
96   TCGA-06-0125   1448.0  1.0                          glioblastoma   
422  TCGA-19-1389    141.0  1.0                          glioblastoma   
369  TCGA-14-1395     42.0  1.0                          glioblastoma   
458  TCGA-26-1442    953.0  0.0                           astrocytoma   

    cg27423208 cg14928902 cg01783151 cg006

In [ ]:
import pandas as pd

# Load the final subset features data
final_df = pd.read_csv("/Users/nijat/Desktop/Final/subset_features_data.csv")
print("Final DataFrame loaded with shape:", final_df.shape)

# Load the CNV cleaned data
cnv_df = pd.read_csv("/Users/nijat/Downloads/Glioma_CNV_Data_2025/methylation/CNV_final_results/cnv_final_cleaned.csv")
print("CNV DataFrame loaded with shape:", cnv_df.shape)

# Merge the data frames on the shared identifier:
# "Patient_ID" from final_df and "Sample_Name" from cnv_df.
merged_df = pd.merge(final_df, cnv_df, left_on='Patient_ID', right_on='Sample_Name', how='inner')

print("Merged DataFrame shape:", merged_df.shape)
print("Total rows:", merged_df.shape[0], "and total columns:", merged_df.shape[1])


Final DataFrame loaded with shape: (619, 184)
CNV DataFrame loaded with shape: (656, 21)
Merged DataFrame shape: (619, 205)
Total rows: 619 and total columns: 205


In [ ]:

print("Merged DataFrame shape:", merged_df.shape)
print("Total rows:", merged_df.shape[0], "and total columns:", merged_df.shape[1])


output_path = "/Users/nijat/Desktop/Final/merged_survival_analysis.csv"
merged_df.to_csv(output_path, index=False)
print(f"Merged DataFrame saved to '{output_path}'.")


Merged DataFrame shape: (619, 205)
Total rows: 619 and total columns: 205
Merged DataFrame saved to '/Users/nijat/Desktop/Final/merged_survival_analysis.csv'.
